***Medical AI report Analyzer using Opensource Models- Llama3.2 and Medgamma:4b***

# **🧬 Medical AI report Analyzer**
Medical AI report Analyzer is an AI-powered healthcare document analysis system that allows users to upload laboratory or medical reports in PDF format and receive structured AI-generated insights in real time. The platform uses OCR technology through ## EasyOCR to extract text from scanned medical documents, followed by medical reasoning using ## MedGemma running locally via ## Ollama. A secondary summarization stage powered by ## Llama 3.2 formats the analysis into a professional and patient-friendly healthcare report.

The system features a lightweight interactive UI built with ## Streamlit, including a real-time workflow status indicator showing each stage of execution such as OCR processing, AI analysis, summarization, and PDF generation. The final AI-generated insights can be downloaded as both Markdown and PDF reports using ## ReportLab.

Designed to run efficiently on Google Colab with ngrok-based public deployment, the platform focuses on low-VRAM medical AI inference, sequential model execution, and automated healthcare report generation while maintaining a simple and scalable architecture for future healthcare AI applications.

In [1]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

!pip install -q \
streamlit \
pyngrok \
requests \
easyocr \
pdf2image \
pymupdf \
reportlab \
markdown2 \
torch \
torchvision \
torchaudio

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (13.5 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently 

In [2]:
!apt-get update
!apt-get install -y poppler-utils

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,608 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [90.8 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dri

In [3]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("Starting Ollama...")

time.sleep(10)

print("Ollama started.")

Starting Ollama...
Ollama started.


In [4]:
!ollama pull medgemma:4b
!ollama pull llama3.2:3b

In [5]:
from google.colab import userdata

from pyngrok import ngrok

NGROK_AUTH_TOKEN =userdata.get("NGROK_AUTH_TOKEN")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("Ngrok authenticated successfully")

Ngrok authenticated successfully


In [6]:
%%writefile app.py
import streamlit as st
import requests
import subprocess
import torch
import gc
import time
import easyocr

from pathlib import Path
from pdf2image import convert_from_path

from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)

from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter


# =========================================================
# CONFIG
# =========================================================

OLLAMA_URL = "http://localhost:11434/api/generate"

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

ANALYSIS_MD = OUTPUT_DIR / "analysis.md"

PDF_OUTPUT = OUTPUT_DIR / "medical_ai_report.pdf"


# =========================================================
# STREAMLIT CONFIG
# =========================================================

st.set_page_config(
    page_title="Medical AI Insights",
    layout="wide"
)


# =========================================================
# SESSION STATE
# =========================================================

if "final_report" not in st.session_state:
    st.session_state.final_report = None

if "analysis_complete" not in st.session_state:
    st.session_state.analysis_complete = False


# =========================================================
# CSS
# =========================================================

st.markdown(
    """
    <style>

    .stApp{
        background:#020c2b;
        color:white;
    }

    .title{
        text-align:center;
        font-size:42px;
        font-weight:700;
        margin-bottom:25px;
        color:#38bdf8;
    }

    .report-box{
        padding:20px;
        border-radius:15px;
        background:#0f172a;
        border:1px solid rgba(255,255,255,0.05);
    }

    </style>
    """,
    unsafe_allow_html=True
)


# =========================================================
# HEADER
# =========================================================

st.markdown(
    """
    <div class="title">
        🧬 Medical AI Insights Platform
    </div>
    """,
    unsafe_allow_html=True
)


# =========================================================
# FLOW UI
# =========================================================

agents = [
    "OCR",
    "MedGemma",
    "Summarizer",
    "PDF"
]

flow_placeholder = st.empty()


def render_flow(current_index=-1, error_index=None):

    with flow_placeholder.container():

        cols = st.columns(len(agents))

        for i, agent in enumerate(agents):

            color = "#334155"

            text_color = "white"

            if error_index == i:
                color = "#dc2626"

            elif i < current_index:
                color = "#16a34a"

            elif i == current_index:
                color = "#facc15"
                text_color = "black"

            cols[i].markdown(
                f"""
                <div style="
                    background:{color};
                    padding:10px;
                    border-radius:10px;
                    text-align:center;
                    font-weight:bold;
                    color:{text_color};
                    font-size:14px;
                    margin-bottom:10px;
                ">
                    {agent}
                </div>
                """,
                unsafe_allow_html=True
            )


render_flow()


# =========================================================
# MODEL FUNCTION
# =========================================================

def run_model(model, prompt):

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )

    return response.json().get("response", "")


# =========================================================
# CLEANUP
# =========================================================

def cleanup_model(model_name):

    try:
        subprocess.run(
            ["ollama", "stop", model_name],
            check=False
        )
    except:
        pass

    try:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    except:
        pass

    gc.collect()

    time.sleep(2)


# =========================================================
# FILE UPLOAD
# =========================================================

uploaded_file = st.file_uploader(
    "Upload Medical Report",
    type=["pdf"]
)


# =========================================================
# MAIN EXECUTION
# =========================================================

if uploaded_file:

    pdf_path = OUTPUT_DIR / uploaded_file.name

    with open(pdf_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    st.success(f"Uploaded: {uploaded_file.name}")

    if st.button("🚀 Generate AI Insights"):

        progress_bar = st.progress(0)

        logs = []

        log_placeholder = st.empty()

        try:

            # =====================================================
            # OCR
            # =====================================================

            render_flow(0)

            logs.append("[INFO] OCR Running")

            log_placeholder.code("\n".join(logs))

            progress_bar.progress(20)

            reader = easyocr.Reader(['en'])

            pages = convert_from_path(str(pdf_path))

            extracted_text = []

            for i, page in enumerate(pages):

                image_path = f"page_{i}.jpg"

                page.save(image_path, "JPEG")

                results = reader.readtext(image_path)

                for detection in results:

                    extracted_text.append(detection[1])

            medical_report_text = "\n".join(extracted_text)

            # =====================================================
            # MEDGEMMA
            # =====================================================

            render_flow(1)

            logs.append("[INFO] MedGemma Running")

            log_placeholder.code("\n".join(logs))

            progress_bar.progress(50)

            med_prompt = f'''
            Analyze this medical report.

            REPORT:
            {medical_report_text}

            Generate:
            - key findings
            - abnormalities
            - medical insights
            - recommendations

            Avoid diagnosis.
            '''

            med_output = run_model(
                "medgemma:4b",
                med_prompt
            )

            cleanup_model("medgemma:4b")

            # =====================================================
            # SUMMARIZER
            # =====================================================

            render_flow(2)

            logs.append("[INFO] Summarizer Running")

            log_placeholder.code("\n".join(logs))

            progress_bar.progress(75)

            summary_prompt = f'''
            Format this into a professional healthcare report.

            CONTENT:
            {med_output}

            Generate:
            - Executive Summary
            - Key Findings
            - Recommendations
            - Patient Friendly Summary
            '''

            st.session_state.final_report = run_model(
                "llama3.2:3b",
                summary_prompt
            )

            final_report = st.session_state.final_report

            cleanup_model("llama3.2:3b")

            # =====================================================
            # SAVE MARKDOWN
            # =====================================================

            with open(ANALYSIS_MD, "w") as f:

                f.write("# Medical AI Insights Report\n\n")

                f.write(final_report)

            # =====================================================
            # DISPLAY REPORT
            # =====================================================

            render_flow(3)

            progress_bar.progress(100)

            st.session_state.analysis_complete = True

            st.subheader("📋 AI Insights")

            st.markdown(
                f"""
                <div class="report-box">
                {final_report}
                </div>
                """,
                unsafe_allow_html=True
            )

        except Exception as e:

            render_flow(error_index=1)

            st.error(str(e))


# =========================================================
# DOWNLOADS
# =========================================================

if st.session_state.analysis_complete:

    col1, col2 = st.columns(2)

    with col1:

        with open(ANALYSIS_MD, "rb") as f:

            st.download_button(
                "⬇ Download Markdown",
                f,
                file_name="analysis.md"
            )

    with col2:

        doc = SimpleDocTemplate(
            str(PDF_OUTPUT),
            pagesize=letter
        )

        styles = getSampleStyleSheet()

        story = []

        with open(ANALYSIS_MD, "r") as f:

            markdown_content = f.read()

        for line in markdown_content.split("\n"):

            if line.strip():

                story.append(
                    Paragraph(
                        line,
                        styles['BodyText']
                    )
                )

                story.append(
                    Spacer(1, 8)
                )

        doc.build(story)

        with open(PDF_OUTPUT, "rb") as pdf_file:

            st.download_button(
                "⬇ Download PDF",
                pdf_file,
                file_name="medical_ai_report.pdf"
            )

Writing app.py


In [7]:
!python -m streamlit run app.py --server.port 8501 &>/content/logs.txt &

In [8]:
import time

print("Waiting for Streamlit to start...")

time.sleep(15)

Waiting for Streamlit to start...


In [9]:
from pyngrok import ngrok

public_url = ngrok.connect(
    addr=8501,
    proto="http",
    bind_tls=True
)

final_url = (
    public_url.public_url
    + "?ngrok-skip-browser-warning=true"
)

print("\n🚀 STREAMLIT URL:\n")

print(final_url)


🚀 STREAMLIT URL:

https://5935-34-143-146-162.ngrok-free.app?ngrok-skip-browser-warning=true


In [10]:
!tail -f /content/logs.txt


2026-05-06 10:14:20.240 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.143.146.162:8501

Progress: |█████████████████████████████████████████████████-| 99.6% CompleteDownloading recognition model, please wait. This may take several minutes depending upon your network connection.
Progress: |█████████████████████████████████████████████████-| 99.3% Complete⠙ ⠹ ⠹ ⠼ ⠙ ⠹ ^C
